<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

Detection uses PPLCNetV4, a RepLKPAN neck with IntraCL, and the inference DB head. Recognition uses the same backbone with height-only downsampling, then a LightSVTR neck and a CTC head. The NRTR head exists only while training, so it is not in these graphs.

## Blocks

A block is `(kernel, in_channels, out_channels, stride, use_se)`. Stride is a pair when the recognizer downsamples height and keeps the width.

Detection stages `s1` to `s4` are the four maps the neck reads. Total stride at the end of the stem plus `s1` is 4, then each later stage halves both sides.

Recognition never strides in width inside the backbone. Height is reduced twice, by the `(2, 1)` blocks, and the rest of the way by the pool in front of the neck.

In [6]:
from fastcore.test import test_eq

test_eq(DET_BLOCKS["s1"][0], (3, 128, 128, 1, True))
test_eq(REC_BLOCKS["b4"][0][3], (2, 1))
test_eq(len(REC_BLOCKS["b4"]), 7)
REC_BLOCKS["b5"][-1]

(3, 768, 768, 1, False)

## Detector

In [8]:
#| echo: false
#| output: asis
show_doc(DetectorConfig)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/config.py#L71){target="_blank" style="float:right; font-size:smaller"}

### DetectorConfig

```python
def DetectorConfig(
    stem_mid:int=64, stem_out:int=128, fpn_channels:int=256, replk_kernel:int=9, head_channels:int=64,
    se_alpha:float=0.16666666666666666
)->None:
```

*Widths, kernel, and squeeze-excite alpha for the medium detector.*

RepLKPAN uses a 9×9 reparameterized depthwise kernel. The squeeze-excite gate is a hardsigmoid with α = 1/6.

In [9]:
from fastcore.test import test_eq

test_eq(DetectorConfig().fpn_channels, 256)
test_eq(DetectorConfig().se_alpha, 1 / 6)
DetectorConfig()

DetectorConfig(stem_mid=64, stem_out=128, fpn_channels=256, replk_kernel=9, head_channels=64, se_alpha=0.16666666666666666)

## Recognizer

In [11]:
#| echo: false
#| output: asis
show_doc(RecognizerConfig)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/config.py#L85){target="_blank" style="float:right; font-size:smaller"}

### RecognizerConfig

```python
def RecognizerConfig(
    stem_mid:int=64, stem_out:int=128, backbone_channels:int=768, svtr_dim:int=192, svtr_depth:int=2,
    svtr_heads:int=8, mlp_ratio:float=4.0, local_kernel:int=7, num_classes:int=18710, image_height:int=48,
    image_width:int=320, se_alpha:float=0.16666666666666666, bn_eps:float=1e-05, final_norm_eps:float=1e-06
)->None:
```

*Backbone width, SVTR size, and the 48×320 image for the medium recognizer.*

The CTC head has 18710 classes: the PP-OCRv6 dictionary, plus the blank. The recognizer's image is 48×320. Two global transformer blocks run at width 192. The norm after the last block is tighter than the norms inside the blocks, 1e-6 rather than 1e-5.

In [12]:
from fastcore.test import test_eq

rec = RecognizerConfig()
test_eq((rec.image_height, rec.image_width), (48, 320))
test_eq(rec.svtr_dim % rec.svtr_heads, 0)
test_eq(rec.num_classes, 18710)
rec

RecognizerConfig(stem_mid=64, stem_out=128, backbone_channels=768, svtr_dim=192, svtr_depth=2, svtr_heads=8, mlp_ratio=4.0, local_kernel=7, num_classes=18710, image_height=48, image_width=320, se_alpha=0.16666666666666666, bn_eps=1e-05, final_norm_eps=1e-06)

## Device

The reference graphs are float32 on CPU. A compiled cache can be reloaded without rebuilding the graph.